In [ ]:
# Spark SQL Step-by-Step Explanation

# ------------------------------------------------------------
# 1. Import SparkSession
# ------------------------------------------------------------
from pyspark.sql import SparkSession

# SparkSession is the entry point for using Spark.
# It allows us to create DataFrames, run SQL queries,
# and manage Spark applications.


# ------------------------------------------------------------
# 2. Create Spark Session
# ------------------------------------------------------------
spark = (
    SparkSession
    .builder
    .appName("Spark SQL")
    .master("local[*]")
    .enableHiveSupport()
    .config("spark.sql.warehouse.dir", "/data/output/spark-warehouse")
    .getOrCreate()
)

# Explanation:
# .appName("Spark SQL") -> Sets application name.
# .master("local[*]") -> Runs Spark locally using all CPU cores.
# .enableHiveSupport() -> Enables Hive support for tables and metadata.
# .config(...) -> Sets Spark warehouse directory.
# .getOrCreate() -> Creates Spark session if it does not exist.


# ------------------------------------------------------------
# 3. Define Employee Schema
# ------------------------------------------------------------
_schema = """
first_name string,
last_name string,
job_title string,
dob string,
email string,
phone string,
salary double,
department_id int
"""

# Schema defines column names and data types.


# ------------------------------------------------------------
# 4. Read Employee CSV File
# ------------------------------------------------------------
emp = (
    spark.read
    .format("csv")
    .schema(_schema)
    .option("header", True)
    .load("/data/input/employee_records_skewed.csv")
)

# Explanation:
# .format("csv") -> Reads CSV file.
# .schema(_schema) -> Applies predefined schema.
# .option("header", True) -> Uses first row as column names.
# .load(...) -> Loads CSV into DataFrame.


# ------------------------------------------------------------
# 5. Define Department Schema
# ------------------------------------------------------------
_dept_schema = """
department_id int,
department_name string,
description string,
city string,
state string,
country string
"""


# ------------------------------------------------------------
# 6. Read Department CSV File
# ------------------------------------------------------------
dept = (
    spark.read
    .format("csv")
    .schema(_dept_schema)
    .option("header", True)
    .load("/data/input/department_data.csv")
)

# Creates department DataFrame.


# ------------------------------------------------------------
# 7. Check Catalog Implementation
# ------------------------------------------------------------
spark.conf.get("spark.sql.catalogImplementation")

# Checks whether Spark uses:
# - in-memory catalog
# OR
# - Hive catalog


# ------------------------------------------------------------
# 8. Show Available Databases
# ------------------------------------------------------------
db = spark.sql("show databases")
db.show()

# Displays all available databases.


# ------------------------------------------------------------
# 9. Show Tables in Default Database
# ------------------------------------------------------------
spark.sql("show tables in default").show()

# Displays tables available in default database.


# ------------------------------------------------------------
# 10. Create Temporary Views
# ------------------------------------------------------------
emp.createOrReplaceTempView("emp_view")
dept.createOrReplaceTempView("dept_view")

# Temporary views allow SQL queries on DataFrames.


# ------------------------------------------------------------
# 11. Filter Employee Data Using SQL
# ------------------------------------------------------------
emp_filtered = spark.sql("""
    SELECT *
    FROM emp_view
    WHERE department_id = 1
""")

# Filters employees belonging to department_id = 1.


# ------------------------------------------------------------
# 12. Show Filtered Data
# ------------------------------------------------------------
emp_filtered.show()

# Displays filtered records.


# ------------------------------------------------------------
# 13. Extract Year from DOB
# ------------------------------------------------------------
emp_temp = spark.sql("""
    SELECT
        e.*,
        date_format(dob, 'yyyy') AS dob_year
    FROM emp_view e
""")

# date_format() extracts year from DOB column.
# Creates a new column called dob_year.


# ------------------------------------------------------------
# 14. Create Another Temporary View
# ------------------------------------------------------------
emp_temp.createOrReplaceTempView("emp_temp_view")

# Registers transformed DataFrame as temp view.


# ------------------------------------------------------------
# 15. Display Updated Data
# ------------------------------------------------------------
spark.sql("SELECT * FROM emp_temp_view").show()

# Shows employee data with dob_year column.


# ------------------------------------------------------------
# 16. Join Employee and Department Data
# ------------------------------------------------------------
emp_final = spark.sql("""
    SELECT /*+ BROADCAST(d) */
        e.*,
        d.department_name
    FROM emp_view e
    LEFT OUTER JOIN dept_view d
    ON e.department_id = d.department_id
""")

# LEFT OUTER JOIN keeps all employee records.
# BROADCAST hint improves join performance
# by sending smaller department table to all nodes.


# ------------------------------------------------------------
# 17. Display Final Joined Data
# ------------------------------------------------------------
emp_final.show()

# Shows final joined result.


# ------------------------------------------------------------
# 18. Save DataFrame as Spark SQL Table
# ------------------------------------------------------------
emp_final.write.format("parquet").saveAsTable("emp_final")

# Saves DataFrame as managed Spark table.
# Parquet format provides:
# - Fast queries
# - Compression
# - Efficient storage


# ------------------------------------------------------------
# 19. Read Data from Saved Table
# ------------------------------------------------------------
emp_new = spark.sql("SELECT * FROM emp_final")

# Reads saved Spark table.


# ------------------------------------------------------------
# 20. Show Table Data
# ------------------------------------------------------------
emp_new.show()

# Displays stored table records.


# ------------------------------------------------------------
# 21. Describe Table Metadata
# ------------------------------------------------------------
spark.sql("DESCRIBE EXTENDED emp_final").show()

# Displays metadata such as:
# - Column names
# - Data types
# - Storage format
# - Table location
# - Provider
# - Statistics


# ------------------------------------------------------------
# End of Spark SQL Workflow
# ------------------------------------------------------------
# Workflow Summary:
# Create Spark Session
#       ↓
# Read CSV Files
#       ↓
# Create DataFrames
#       ↓
# Create Temp Views
#       ↓
# Run SQL Queries
#       ↓
# Transform Data
#       ↓
# Join Tables
#       ↓
# Save Spark Table
#       ↓
# Read & Describe Table
